# Meta-NCA for 1D-ARC: experiments

Few-shot reasoning on the [1D-ARC](https://github.com/khalil-research/1D-ARC) benchmark with a **Meta-Neural Cellular Automaton**. A support encoder turns the few-shot examples into a rule vector, a hypernetwork turns that vector into task-specific 1D-convolution weights, and an NCA applies those weights to the query input over several steps.

Each code cell below is one complete experiment (dataset, model, training and evaluation). Experiments 1-3 can each run on their own. The final pipeline cell reuses the imports from the first cell. Data is read from `../data/1D-ARC/`, so run the notebook from the `notebooks/` directory.

## Experiment 1: Raw Meta-NCA (no colour abstraction)
The model operates directly on raw colour indices (0-9).

In [1]:
import os
import json
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LinearLR, ExponentialLR, ChainedScheduler

# ==========================================
# 1. DATASET 
# ==========================================

class RealARCDataset(Dataset):
    def __init__(self, json_dir, max_seq_len=30, pad_idx=11, num_support=30):
        self.json_dir = json_dir
        self.files = [f for f in os.listdir(json_dir) if f.endswith('.json')]
        self.max_seq_len = max_seq_len
        self.pad_idx = pad_idx
        self.num_support = num_support

    def __len__(self):
        return len(self.files)

    def flatten_to_1d(self, grid):
        if not isinstance(grid, list):
            return [grid]
        flat_list = []
        for item in grid:
            if isinstance(item, list):
                flat_list.extend(self.flatten_to_1d(item))
            else:
                flat_list.append(item)
        return flat_list

    def pad_sequence(self, seq_raw):
        seq = self.flatten_to_1d(seq_raw)
        if len(seq) >= self.max_seq_len:
            return seq[:self.max_seq_len]
        return seq + [self.pad_idx] * (self.max_seq_len - len(seq))

    def __getitem__(self, idx):
        file_path = os.path.join(self.json_dir, self.files[idx])
        with open(file_path, 'r') as f:
            task_data = json.load(f)

        raw_support = task_data['train']
        augmented_support = (raw_support * 15)[:self.num_support]

        support_in = torch.tensor([self.pad_sequence(pair['input']) for pair in augmented_support], dtype=torch.long)
        support_out = torch.tensor([self.pad_sequence(pair['output']) for pair in augmented_support], dtype=torch.long)
        support_mask = (support_in == self.pad_idx)

        raw_query = task_data['test'][0] 
        query_in = torch.tensor(self.pad_sequence(raw_query['input']), dtype=torch.long)
        query_target = torch.tensor(self.pad_sequence(raw_query['output']), dtype=torch.long)
        query_mask = (query_in == self.pad_idx)

        return {
            'support_in': support_in,
            'support_out': support_out,
            'support_mask': support_mask,
            'query_in': query_in,
            'query_target': query_target,
            'query_mask': query_mask
        }

# ==========================================
# 2. RAW META-NCA ARCHITECTURE (No Abstraction)
# ==========================================

class SupportEncoder(nn.Module):
    def __init__(self, vocab_size=12, d_model=64, pad_idx=11):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        
        self.feature_extractor = nn.Sequential(
            nn.Conv1d(d_model * 2, d_model * 2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(d_model * 2, d_model, kernel_size=3, padding=1),
            nn.AdaptiveMaxPool1d(1) 
        )
        
        self.rule_projector = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

    def forward(self, support_in, support_out):
        B, N, S = support_in.shape
        
        emb_in = self.embedding(support_in)
        emb_out = self.embedding(support_out)
        
        combined = torch.cat([emb_in, emb_out], dim=-1)
        combined = combined.view(B * N, S, -1).transpose(1, 2)
        
        features = self.feature_extractor(combined).squeeze(-1)
        features = features.view(B, N, -1)
        
        rule_vector = features.mean(dim=1) 
        return self.rule_projector(rule_vector)

class MetaNCA(nn.Module):
    def __init__(self, vocab_size=12, d_model=64, nca_steps=10, pad_idx=11):
        super().__init__()
        self.d_model = d_model
        self.nca_steps = nca_steps
        self.pad_idx = pad_idx
        
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.support_encoder = SupportEncoder(vocab_size, d_model, pad_idx)
        
        self.kernel_size = 3
        weight_params = d_model * d_model * self.kernel_size
        bias_params = d_model
        
        self.hyper_net = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.ReLU(),
            nn.Linear(d_model * 2, weight_params + bias_params)
        )
        self.output_layer = nn.Linear(d_model, vocab_size)

    def forward(self, support_in, support_out, support_mask, query_in, query_mask, nca_steps=None):
        B, S = query_in.shape
        steps = nca_steps if nca_steps is not None else self.nca_steps
        
        # 1. Extract the rule from the raw colors
        rule_vector = self.support_encoder(support_in, support_out)
        
        # 2. Generate dynamics weights based on those specific colors
        generated_params = self.hyper_net(rule_vector)
        
        weight_size = self.d_model * self.d_model * self.kernel_size
        weights_flat = generated_params[:, :weight_size]
        biases_flat = generated_params[:, weight_size:]
        
        dynamic_weights = weights_flat.reshape(B * self.d_model, self.d_model, self.kernel_size)
        dynamic_biases = biases_flat.reshape(B * self.d_model)
        
        # 3. Initialize query state
        state = self.embedding(query_in).transpose(1, 2)
        
        # 4. Execute standard recursive physics
        for _ in range(steps):
            state_reshaped = state.reshape(1, B * self.d_model, S)
            update = F.conv1d(
                state_reshaped, weight=dynamic_weights, bias=dynamic_biases, 
                padding=self.kernel_size // 2, groups=B 
            )
            update = update.view(B, self.d_model, S)
            state = state + F.relu(update)
            
        final_state = state.transpose(1, 2)
        return self.output_layer(final_state)

# ==========================================
# 3. TRAINING & EVALUATION PROTOCOLS
# ==========================================

def train_epoch(model, dataloader, optimizer, criterion, device, max_grad_norm=1.0):
    model.train()
    total_loss = 0

    for batch in dataloader:
        optimizer.zero_grad()

        # Load raw data straight to device (No abstraction wrapper)
        support_in = batch['support_in'].to(device)
        support_out = batch['support_out'].to(device)
        support_mask = batch['support_mask'].to(device)
        query_in = batch['query_in'].to(device)
        query_target = batch['query_target'].to(device)
        query_mask = batch['query_mask'].to(device)

        dynamic_steps = random.randint(8, 12)

        # Forward pass on raw colors
        logits = model(
            support_in, support_out, support_mask, 
            query_in, query_mask, 
            nca_steps=dynamic_steps
        )

        flat_logits = logits.reshape(-1, logits.size(-1))
        flat_targets = query_target.reshape(-1) 
        
        loss = criterion(flat_logits, flat_targets)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(dataloader)


def evaluate_tasks(model, dataloader, device):
    model.eval()
    correct_tasks = 0
    total_tasks = 0

    with torch.no_grad():
        for batch in dataloader:
            support_in = batch['support_in'].to(device)
            support_out = batch['support_out'].to(device)
            support_mask = batch['support_mask'].to(device)
            query_in = batch['query_in'].to(device)
            query_target = batch['query_target'].to(device)
            query_mask = batch['query_mask'].to(device)

            # Evaluate on raw colors
            logits = model(
                support_in, support_out, support_mask, 
                query_in, query_mask, 
                nca_steps=10 
            )
            
            predictions = logits.argmax(dim=-1) 

            for i in range(len(predictions)):
                valid_mask = ~query_mask[i] 
                
                valid_preds = predictions[i][valid_mask]
                valid_targets = query_target[i][valid_mask]

                if torch.equal(valid_preds, valid_targets):
                    correct_tasks += 1
                total_tasks += 1

    return (correct_tasks / total_tasks) * 100 if total_tasks > 0 else 0.0

# ==========================================
# 4. MAIN EXECUTION
# ==========================================

if __name__ == "__main__":
    os.environ['WANDB_MODE'] = 'offline' 
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Initializing Raw Meta-NCA (No Abstraction) on: {device}")

    # Use your predefined directories
    TRAIN_DIR = r"../data/1D-ARC/train"
    TEST_DIR_SAME = r"../data/1D-ARC/test_same_concept"
    TEST_DIR_CROSS = r"../data/1D-ARC/test_cross_concept"
    
    BATCH_SIZE = 16
    MAX_SEQ_LEN = 30
    PAD_IDX = 11
    EPOCHS = 100 

    print("Loading datasets...")
    train_dataset = RealARCDataset(TRAIN_DIR, max_seq_len=MAX_SEQ_LEN, pad_idx=PAD_IDX)
    test_dataset = RealARCDataset(TEST_DIR_SAME, max_seq_len=MAX_SEQ_LEN, pad_idx=PAD_IDX)
    test_cross_dataset = RealARCDataset(TEST_DIR_CROSS, max_seq_len=MAX_SEQ_LEN, pad_idx=PAD_IDX)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_cross_loader = DataLoader(test_cross_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # Standard vocab_size=12 for raw colors
    model = MetaNCA(vocab_size=12, d_model=64, nca_steps=10, pad_idx=PAD_IDX).to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    warmup_scheduler = LinearLR(optimizer, start_factor=0.1, total_iters=10)
    decay_scheduler = ExponentialLR(optimizer, gamma=0.95)
    scheduler = ChainedScheduler([warmup_scheduler, decay_scheduler])

    print("Beginning raw training protocol...")
    best_acc = 0.0

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device, max_grad_norm=1.0)
        scheduler.step() 
        current_lr = optimizer.param_groups[0]['lr']
        
        if epoch % 5 == 0:
            test_acc = evaluate_tasks(model, test_loader, device)
            if test_acc > best_acc:
                best_acc = test_acc
                torch.save(model.state_dict(), "best_raw_nca.pth")
                marker = "*"
            else:
                marker = ""
                
            print(f"Epoch {epoch:03d}/{EPOCHS} | LR: {current_lr:.5f} | Loss: {train_loss:.4f} | Same-Concept Acc: {test_acc:.2f}% {marker}")

    # Final Evaluation Block
    print("\n--- Final Cross-Concept Evaluation ---")
    best_model = MetaNCA(vocab_size=12, d_model=64, nca_steps=10, pad_idx=PAD_IDX).to(device)
    best_model.load_state_dict(torch.load("best_raw_nca.pth", map_location=device))
    
    cross_test_acc = evaluate_tasks(best_model, test_cross_loader, device)
    
    print(f"========================================")
    print(f" Raw Meta-NCA Same-Concept Acc:  {best_acc:.2f}%")
    print(f" Raw Meta-NCA Cross-Concept Acc: {cross_test_acc:.2f}%")
    print(f"========================================")

Initializing Raw Meta-NCA (No Abstraction) on: cuda
Loading datasets...
Beginning raw training protocol...
Epoch 005/100 | LR: 0.00043 | Loss: 0.6194 | Same-Concept Acc: 9.33% *
Epoch 010/100 | LR: 0.00060 | Loss: 0.4146 | Same-Concept Acc: 18.67% *
Epoch 015/100 | LR: 0.00046 | Loss: 0.3111 | Same-Concept Acc: 49.33% *
Epoch 020/100 | LR: 0.00036 | Loss: 0.2208 | Same-Concept Acc: 49.33% 
Epoch 025/100 | LR: 0.00028 | Loss: 0.1477 | Same-Concept Acc: 49.33% 
Epoch 030/100 | LR: 0.00021 | Loss: 0.1005 | Same-Concept Acc: 56.00% *
Epoch 035/100 | LR: 0.00017 | Loss: 0.0830 | Same-Concept Acc: 54.67% 
Epoch 040/100 | LR: 0.00013 | Loss: 0.0624 | Same-Concept Acc: 52.00% 
Epoch 045/100 | LR: 0.00010 | Loss: 0.0461 | Same-Concept Acc: 53.33% 
Epoch 050/100 | LR: 0.00008 | Loss: 0.0371 | Same-Concept Acc: 52.00% 
Epoch 055/100 | LR: 0.00006 | Loss: 0.0246 | Same-Concept Acc: 54.67% 
Epoch 060/100 | LR: 0.00005 | Loss: 0.0250 | Same-Concept Acc: 52.00% 
Epoch 065/100 | LR: 0.00004 | Loss: 0.

## Experiment 2: Meta-NCA + dynamic colour abstraction
Same architecture and hyperparameters as Experiment 1. Before the forward pass, each task's colours are remapped to abstract role IDs (`apply_abstract_color_mapping`).

In [3]:
import os
import json
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LinearLR, ExponentialLR, ChainedScheduler

# ==========================================
# 1. DATASET & DYNAMIC ABSTRACTION
# ==========================================

class RealARCDataset(Dataset):
    def __init__(self, json_dir, max_seq_len=30, pad_idx=11, num_support=30):
        self.json_dir = json_dir
        self.files = [f for f in os.listdir(json_dir) if f.endswith('.json')]
        self.max_seq_len = max_seq_len
        self.pad_idx = pad_idx
        self.num_support = num_support

    def __len__(self):
        return len(self.files)

    def flatten_to_1d(self, grid):
        if not isinstance(grid, list):
            return [grid]
        flat_list = []
        for item in grid:
            if isinstance(item, list):
                flat_list.extend(self.flatten_to_1d(item))
            else:
                flat_list.append(item)
        return flat_list

    def pad_sequence(self, seq_raw):
        seq = self.flatten_to_1d(seq_raw)
        if len(seq) >= self.max_seq_len:
            return seq[:self.max_seq_len]
        return seq + [self.pad_idx] * (self.max_seq_len - len(seq))

    def __getitem__(self, idx):
        file_path = os.path.join(self.json_dir, self.files[idx])
        with open(file_path, 'r') as f:
            task_data = json.load(f)

        raw_support = task_data['train']
        augmented_support = (raw_support * 15)[:self.num_support]

        support_in = torch.tensor([self.pad_sequence(pair['input']) for pair in augmented_support], dtype=torch.long)
        support_out = torch.tensor([self.pad_sequence(pair['output']) for pair in augmented_support], dtype=torch.long)
        support_mask = (support_in == self.pad_idx)

        raw_query = task_data['test'][0] 
        query_in = torch.tensor(self.pad_sequence(raw_query['input']), dtype=torch.long)
        query_target = torch.tensor(self.pad_sequence(raw_query['output']), dtype=torch.long)
        query_mask = (query_in == self.pad_idx)

        return {
            'support_in': support_in,
            'support_out': support_out,
            'support_mask': support_mask,
            'query_in': query_in,
            'query_target': query_target,
            'query_mask': query_mask
        }

def apply_abstract_color_mapping(support_in, support_out, query_in, query_target=None, pad_idx=11, bg_idx=0):
    """
    Dynamically remaps the colors in a task to an abstract, relative palette.
    Removes visual bias so the model focuses purely on algorithmic logic.
    """
    B = support_in.shape[0]
    
    abs_sup_in = torch.full_like(support_in, bg_idx)
    abs_sup_out = torch.full_like(support_out, bg_idx)
    abs_query_in = torch.full_like(query_in, bg_idx)
    abs_query_target = torch.full_like(query_target, bg_idx) if query_target is not None else None

    for b in range(B):
        # Find unique foreground colors in the support inputs
        task_pixels = support_in[b].flatten()
        unique_colors = torch.unique(task_pixels)
        valid_colors = [c.item() for c in unique_colors if c.item() not in [bg_idx, pad_idx]]
        
        # Build mapping: map raw foreground colors to 1, 2, 3...
        forward_map = {bg_idx: bg_idx, pad_idx: pad_idx}
        for abstract_id, raw_color in enumerate(valid_colors, start=1):
            forward_map[raw_color] = abstract_id
            
        # Apply the mapping
        for raw, abstract in forward_map.items():
            abs_sup_in[b][support_in[b] == raw] = abstract
            abs_sup_out[b][support_out[b] == raw] = abstract
            abs_query_in[b][query_in[b] == raw] = abstract
            if abs_query_target is not None:
                abs_query_target[b][query_target[b] == raw] = abstract

    return abs_sup_in, abs_sup_out, abs_query_in, abs_query_target


# ==========================================
# 2. META-NCA ARCHITECTURE
# ==========================================

class SupportEncoder(nn.Module):
    def __init__(self, vocab_size=12, d_model=64, pad_idx=11):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        
        self.feature_extractor = nn.Sequential(
            nn.Conv1d(d_model * 2, d_model * 2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(d_model * 2, d_model, kernel_size=3, padding=1),
            nn.AdaptiveMaxPool1d(1) 
        )
        
        self.rule_projector = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

    def forward(self, support_in, support_out):
        B, N, S = support_in.shape
        emb_in = self.embedding(support_in)
        emb_out = self.embedding(support_out)
        
        combined = torch.cat([emb_in, emb_out], dim=-1)
        combined = combined.view(B * N, S, -1).transpose(1, 2)
        
        features = self.feature_extractor(combined).squeeze(-1)
        features = features.view(B, N, -1)
        
        rule_vector = features.mean(dim=1) 
        return self.rule_projector(rule_vector)

class MetaNCA(nn.Module):
    def __init__(self, vocab_size=12, d_model=64, nca_steps=10, pad_idx=11):
        super().__init__()
        self.d_model = d_model
        self.nca_steps = nca_steps
        self.pad_idx = pad_idx
        
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.support_encoder = SupportEncoder(vocab_size, d_model, pad_idx)
        
        self.kernel_size = 3
        weight_params = d_model * d_model * self.kernel_size
        bias_params = d_model
        
        self.hyper_net = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.ReLU(),
            nn.Linear(d_model * 2, weight_params + bias_params)
        )
        self.output_layer = nn.Linear(d_model, vocab_size)

    def forward(self, support_in, support_out, support_mask, query_in, query_mask, nca_steps=None):
        B, S = query_in.shape
        steps = nca_steps if nca_steps is not None else self.nca_steps
        
        rule_vector = self.support_encoder(support_in, support_out)
        generated_params = self.hyper_net(rule_vector)
        
        weight_size = self.d_model * self.d_model * self.kernel_size
        weights_flat = generated_params[:, :weight_size]
        biases_flat = generated_params[:, weight_size:]
        
        dynamic_weights = weights_flat.reshape(B * self.d_model, self.d_model, self.kernel_size)
        dynamic_biases = biases_flat.reshape(B * self.d_model)
        
        state = self.embedding(query_in).transpose(1, 2)
        
        for _ in range(steps):
            state_reshaped = state.reshape(1, B * self.d_model, S)
            update = F.conv1d(
                state_reshaped, weight=dynamic_weights, bias=dynamic_biases, 
                padding=self.kernel_size // 2, groups=B 
            )
            update = update.view(B, self.d_model, S)
            state = state + F.relu(update)
            
        final_state = state.transpose(1, 2)
        return self.output_layer(final_state)


# ==========================================
# 3. TRAINING & EVALUATION PROTOCOLS
# ==========================================

def train_epoch(model, dataloader, optimizer, criterion, device, pad_idx, max_grad_norm=1.0):
    model.train()
    total_loss = 0

    for batch in dataloader:
        optimizer.zero_grad()

        # Load to device
        support_in = batch['support_in'].to(device)
        support_out = batch['support_out'].to(device)
        support_mask = batch['support_mask'].to(device)
        query_in = batch['query_in'].to(device)
        query_target = batch['query_target'].to(device)
        query_mask = batch['query_mask'].to(device)

        # CRITICAL ADDITION: Dynamic Color Abstraction Wrapper
        abs_sup_in, abs_sup_out, abs_query_in, abs_query_target = apply_abstract_color_mapping(
            support_in, support_out, query_in, query_target, pad_idx=pad_idx
        )

        dynamic_steps = random.randint(8, 12)

        # Forward pass using ABSTRACTED data
        logits = model(
            abs_sup_in, abs_sup_out, support_mask, 
            abs_query_in, query_mask, 
            nca_steps=dynamic_steps
        )

        flat_logits = logits.reshape(-1, logits.size(-1))
        flat_targets = abs_query_target.reshape(-1) # Compute loss against abstracted targets!
        
        loss = criterion(flat_logits, flat_targets)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(dataloader)


def evaluate_tasks(model, dataloader, device, pad_idx):
    model.eval()
    correct_tasks = 0
    total_tasks = 0

    with torch.no_grad():
        for batch in dataloader:
            support_in = batch['support_in'].to(device)
            support_out = batch['support_out'].to(device)
            support_mask = batch['support_mask'].to(device)
            query_in = batch['query_in'].to(device)
            query_target = batch['query_target'].to(device)
            query_mask = batch['query_mask'].to(device)

            # Abstract the evaluation data as well
            abs_sup_in, abs_sup_out, abs_query_in, abs_query_target = apply_abstract_color_mapping(
                support_in, support_out, query_in, query_target, pad_idx=pad_idx
            )

            logits = model(
                abs_sup_in, abs_sup_out, support_mask, 
                abs_query_in, query_mask, 
                nca_steps=10 
            )
            
            predictions = logits.argmax(dim=-1) 

            for i in range(len(predictions)):
                valid_mask = ~query_mask[i] 
                
                # Compare abstract predictions against abstract targets
                valid_preds = predictions[i][valid_mask]
                valid_targets = abs_query_target[i][valid_mask]

                if torch.equal(valid_preds, valid_targets):
                    correct_tasks += 1
                total_tasks += 1

    return (correct_tasks / total_tasks) * 100 if total_tasks > 0 else 0.0


# ==========================================
# 4. MAIN EXECUTION
# ==========================================

if __name__ == "__main__":
    os.environ['WANDB_MODE'] = 'offline' 
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Initializing Abstraction-Wrapped Meta-NCA on: {device}")

    # Use your predefined directories
    TRAIN_DIR = r"../data/1D-ARC/train"
    TEST_DIR_SAME = r"../data/1D-ARC/test_same_concept"
    TEST_DIR_CROSS = r"../data/1D-ARC/test_cross_concept"
    
    BATCH_SIZE = 16
    MAX_SEQ_LEN = 30
    PAD_IDX = 11
    EPOCHS = 100 

    print("Loading datasets...")
    train_dataset = RealARCDataset(TRAIN_DIR, max_seq_len=MAX_SEQ_LEN, pad_idx=PAD_IDX)
    test_dataset = RealARCDataset(TEST_DIR_SAME, max_seq_len=MAX_SEQ_LEN, pad_idx=PAD_IDX)
    test_cross_dataset = RealARCDataset(TEST_DIR_CROSS, max_seq_len=MAX_SEQ_LEN, pad_idx=PAD_IDX)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_cross_loader = DataLoader(test_cross_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = MetaNCA(vocab_size=12, d_model=64, nca_steps=10, pad_idx=PAD_IDX).to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    warmup_scheduler = LinearLR(optimizer, start_factor=0.1, total_iters=10)
    decay_scheduler = ExponentialLR(optimizer, gamma=0.95)
    scheduler = ChainedScheduler([warmup_scheduler, decay_scheduler])

    print("Beginning training protocol...")
    best_acc = 0.0

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device, PAD_IDX, max_grad_norm=1.0)
        scheduler.step() 
        current_lr = optimizer.param_groups[0]['lr']
        
        if epoch % 5 == 0:
            test_acc = evaluate_tasks(model, test_loader, device, PAD_IDX)
            if test_acc > best_acc:
                best_acc = test_acc
                torch.save(model.state_dict(), "best_abstract_nca.pth")
                marker = "*"
            else:
                marker = ""
                
            print(f"Epoch {epoch:03d}/{EPOCHS} | LR: {current_lr:.5f} | Loss: {train_loss:.4f} | Same-Concept Acc: {test_acc:.2f}% {marker}")

    # Final Evaluation Block
    print("\n--- Final Cross-Concept Evaluation ---")
    best_model = MetaNCA(vocab_size=12, d_model=64, nca_steps=10, pad_idx=PAD_IDX).to(device)
    best_model.load_state_dict(torch.load("best_abstract_nca.pth", map_location=device))
    
    cross_test_acc = evaluate_tasks(best_model, test_cross_loader, device, PAD_IDX)
    
    print(f"========================================")
    print(f" Abstract Meta-NCA Same-Concept Acc:  {best_acc:.2f}%")
    print(f" Abstract Meta-NCA Cross-Concept Acc: {cross_test_acc:.2f}%")
    print(f"========================================")

Initializing Abstraction-Wrapped Meta-NCA on: cuda
Loading datasets...
Beginning training protocol...
Epoch 005/100 | LR: 0.00043 | Loss: 0.2092 | Same-Concept Acc: 60.00% *
Epoch 010/100 | LR: 0.00060 | Loss: 0.1471 | Same-Concept Acc: 69.33% *
Epoch 015/100 | LR: 0.00046 | Loss: 0.0916 | Same-Concept Acc: 84.00% *
Epoch 020/100 | LR: 0.00036 | Loss: 0.0731 | Same-Concept Acc: 80.00% 
Epoch 025/100 | LR: 0.00028 | Loss: 0.0511 | Same-Concept Acc: 84.00% 
Epoch 030/100 | LR: 0.00021 | Loss: 0.0402 | Same-Concept Acc: 86.67% *
Epoch 035/100 | LR: 0.00017 | Loss: 0.0305 | Same-Concept Acc: 85.33% 
Epoch 040/100 | LR: 0.00013 | Loss: 0.0179 | Same-Concept Acc: 85.33% 
Epoch 045/100 | LR: 0.00010 | Loss: 0.0163 | Same-Concept Acc: 88.00% *
Epoch 050/100 | LR: 0.00008 | Loss: 0.0093 | Same-Concept Acc: 86.67% 
Epoch 055/100 | LR: 0.00006 | Loss: 0.0077 | Same-Concept Acc: 85.33% 
Epoch 060/100 | LR: 0.00005 | Loss: 0.0063 | Same-Concept Acc: 86.67% 
Epoch 065/100 | LR: 0.00004 | Loss: 0.005

## Experiment 3: Scaled Meta-NCA (d_model = 256)
A larger model: d_model = 256, 30 NCA steps (25-35 during training), dropout, and 150 epochs. **The logged output stops at epoch 15** because the run was interrupted, so this notebook does not contain final numbers for this configuration (see the README).

In [ ]:
import os
import json
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LinearLR, ExponentialLR, ChainedScheduler

# ==========================================
# 1. DATASET & DYNAMIC ABSTRACTION
# ==========================================

class RealARCDataset(Dataset):
    def __init__(self, json_dir, max_seq_len=30, pad_idx=11, num_support=30):
        self.json_dir = json_dir
        self.files = [f for f in os.listdir(json_dir) if f.endswith('.json')]
        self.max_seq_len = max_seq_len
        self.pad_idx = pad_idx
        self.num_support = num_support

    def __len__(self):
        return len(self.files)

    def flatten_to_1d(self, grid):
        if not isinstance(grid, list):
            return [grid]
        flat_list = []
        for item in grid:
            if isinstance(item, list):
                flat_list.extend(self.flatten_to_1d(item))
            else:
                flat_list.append(item)
        return flat_list

    def pad_sequence(self, seq_raw):
        seq = self.flatten_to_1d(seq_raw)
        if len(seq) >= self.max_seq_len:
            return seq[:self.max_seq_len]
        return seq + [self.pad_idx] * (self.max_seq_len - len(seq))

    def __getitem__(self, idx):
        file_path = os.path.join(self.json_dir, self.files[idx])
        with open(file_path, 'r') as f:
            task_data = json.load(f)

        raw_support = task_data['train']
        augmented_support = (raw_support * 15)[:self.num_support]

        support_in = torch.tensor([self.pad_sequence(pair['input']) for pair in augmented_support], dtype=torch.long)
        support_out = torch.tensor([self.pad_sequence(pair['output']) for pair in augmented_support], dtype=torch.long)
        support_mask = (support_in == self.pad_idx)

        raw_query = task_data['test'][0] 
        query_in = torch.tensor(self.pad_sequence(raw_query['input']), dtype=torch.long)
        query_target = torch.tensor(self.pad_sequence(raw_query['output']), dtype=torch.long)
        query_mask = (query_in == self.pad_idx)

        return {
            'support_in': support_in,
            'support_out': support_out,
            'support_mask': support_mask,
            'query_in': query_in,
            'query_target': query_target,
            'query_mask': query_mask
        }

def apply_abstract_color_mapping(support_in, support_out, query_in, query_target=None, pad_idx=11, bg_idx=0):
    B = support_in.shape[0]
    
    abs_sup_in = torch.full_like(support_in, bg_idx)
    abs_sup_out = torch.full_like(support_out, bg_idx)
    abs_query_in = torch.full_like(query_in, bg_idx)
    abs_query_target = torch.full_like(query_target, bg_idx) if query_target is not None else None

    for b in range(B):
        task_pixels = support_in[b].flatten()
        unique_colors = torch.unique(task_pixels)
        valid_colors = [c.item() for c in unique_colors if c.item() not in [bg_idx, pad_idx]]
        
        forward_map = {bg_idx: bg_idx, pad_idx: pad_idx}
        for abstract_id, raw_color in enumerate(valid_colors, start=1):
            forward_map[raw_color] = abstract_id
            
        for raw, abstract in forward_map.items():
            abs_sup_in[b][support_in[b] == raw] = abstract
            abs_sup_out[b][support_out[b] == raw] = abstract
            abs_query_in[b][query_in[b] == raw] = abstract
            if abs_query_target is not None:
                abs_query_target[b][query_target[b] == raw] = abstract

    return abs_sup_in, abs_sup_out, abs_query_in, abs_query_target


# ==========================================
# 2. SCALED META-NCA ARCHITECTURE
# ==========================================

class SupportEncoder(nn.Module):
    def __init__(self, vocab_size=12, d_model=256, pad_idx=11):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        
        self.feature_extractor = nn.Sequential(
            nn.Conv1d(d_model * 2, d_model * 2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(d_model * 2, d_model, kernel_size=3, padding=1),
            nn.AdaptiveMaxPool1d(1) 
        )
        
        # Added Dropout for regularization on the larger representation
        self.rule_projector = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(d_model, d_model)
        )

    def forward(self, support_in, support_out):
        B, N, S = support_in.shape
        emb_in = self.embedding(support_in)
        emb_out = self.embedding(support_out)
        
        combined = torch.cat([emb_in, emb_out], dim=-1)
        combined = combined.view(B * N, S, -1).transpose(1, 2)
        
        features = self.feature_extractor(combined).squeeze(-1)
        features = features.view(B, N, -1)
        
        rule_vector = features.mean(dim=1) 
        return self.rule_projector(rule_vector)

class MetaNCA(nn.Module):
    def __init__(self, vocab_size=12, d_model=256, nca_steps=30, pad_idx=11):
        super().__init__()
        self.d_model = d_model
        self.nca_steps = nca_steps
        self.pad_idx = pad_idx
        
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.support_encoder = SupportEncoder(vocab_size, d_model, pad_idx)
        
        self.kernel_size = 3
        weight_params = d_model * d_model * self.kernel_size
        bias_params = d_model
        
        self.hyper_net = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.LayerNorm(d_model * 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(d_model * 2, d_model * 2),
            nn.LayerNorm(d_model * 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(d_model * 2, weight_params + bias_params)
        )
        
        # --- STABILITY FIX 1: Near-Zero Initialization ---
        # Force the hypernetwork to initially generate weights very close to 0.
        # This prevents the recursive loop from exponentiating noise.
        nn.init.uniform_(self.hyper_net[-1].weight, -1e-4, 1e-4)
        nn.init.zeros_(self.hyper_net[-1].bias)
        
        self.output_layer = nn.Linear(d_model, vocab_size)

    def forward(self, support_in, support_out, support_mask, query_in, query_mask, nca_steps=None):
        B, S = query_in.shape
        steps = nca_steps if nca_steps is not None else self.nca_steps
        
        rule_vector = self.support_encoder(support_in, support_out)
        generated_params = self.hyper_net(rule_vector)
        
        weight_size = self.d_model * self.d_model * self.kernel_size
        weights_flat = generated_params[:, :weight_size]
        biases_flat = generated_params[:, weight_size:]
        
        dynamic_weights = weights_flat.reshape(B * self.d_model, self.d_model, self.kernel_size)
        dynamic_biases = biases_flat.reshape(B * self.d_model)
        
        # --- STABILITY FIX 2: Variance Scaling (Fan-In) ---
        # Scale down the weights so the 256-dim dot products don't blow up
        dynamic_weights = dynamic_weights / (self.d_model ** 0.5)
        
        state = self.embedding(query_in).transpose(1, 2)
        
        for _ in range(steps):
            state_reshaped = state.reshape(1, B * self.d_model, S)
            update = F.conv1d(
                state_reshaped, weight=dynamic_weights, bias=dynamic_biases, 
                padding=self.kernel_size // 2, groups=B 
            )
            update = update.view(B, self.d_model, S)
            
            # Use Tanh instead of ReLU on the update to strictly bound the values between -1 and 1
            state = state + torch.tanh(update)
            
        final_state = state.transpose(1, 2)
        return self.output_layer(final_state)


# ==========================================
# 3. TRAINING & EVALUATION PROTOCOLS
# ==========================================

def train_epoch(model, dataloader, optimizer, criterion, device, pad_idx, max_grad_norm=1.0):
    model.train()
    total_loss = 0

    for batch in dataloader:
        optimizer.zero_grad()

        support_in = batch['support_in'].to(device)
        support_out = batch['support_out'].to(device)
        support_mask = batch['support_mask'].to(device)
        query_in = batch['query_in'].to(device)
        query_target = batch['query_target'].to(device)
        query_mask = batch['query_mask'].to(device)

        abs_sup_in, abs_sup_out, abs_query_in, abs_query_target = apply_abstract_color_mapping(
            support_in, support_out, query_in, query_target, pad_idx=pad_idx
        )

        # Extended randomized steps to allow full-sequence traversal
        dynamic_steps = random.randint(25, 35)

        logits = model(
            abs_sup_in, abs_sup_out, support_mask, 
            abs_query_in, query_mask, 
            nca_steps=dynamic_steps
        )

        flat_logits = logits.reshape(-1, logits.size(-1))
        flat_targets = abs_query_target.reshape(-1)
        
        loss = criterion(flat_logits, flat_targets)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(dataloader)


def evaluate_tasks(model, dataloader, device, pad_idx):
    model.eval()
    correct_tasks = 0
    total_tasks = 0

    with torch.no_grad():
        for batch in dataloader:
            support_in = batch['support_in'].to(device)
            support_out = batch['support_out'].to(device)
            support_mask = batch['support_mask'].to(device)
            query_in = batch['query_in'].to(device)
            query_target = batch['query_target'].to(device)
            query_mask = batch['query_mask'].to(device)

            abs_sup_in, abs_sup_out, abs_query_in, abs_query_target = apply_abstract_color_mapping(
                support_in, support_out, query_in, query_target, pad_idx=pad_idx
            )

            # Evaluate with the scaled-up step count
            logits = model(
                abs_sup_in, abs_sup_out, support_mask, 
                abs_query_in, query_mask, 
                nca_steps=30 
            )
            
            predictions = logits.argmax(dim=-1) 

            for i in range(len(predictions)):
                valid_mask = ~query_mask[i] 
                
                valid_preds = predictions[i][valid_mask]
                valid_targets = abs_query_target[i][valid_mask]

                if torch.equal(valid_preds, valid_targets):
                    correct_tasks += 1
                total_tasks += 1

    return (correct_tasks / total_tasks) * 100 if total_tasks > 0 else 0.0


# ==========================================
# 4. MAIN EXECUTION
# ==========================================

if __name__ == "__main__":
    os.environ['WANDB_MODE'] = 'offline' 
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Initializing Scaled Abstraction-Wrapped Meta-NCA on: {device}")

    TRAIN_DIR = r"../data/1D-ARC/train"
    TEST_DIR_SAME = r"../data/1D-ARC/test_same_concept"
    TEST_DIR_CROSS = r"../data/1D-ARC/test_cross_concept"
    
    # Unleashing the hardware parameters
    BATCH_SIZE = 32
    MAX_SEQ_LEN = 30
    PAD_IDX = 11
    EPOCHS = 150 

    print("Loading datasets...")
    train_dataset = RealARCDataset(TRAIN_DIR, max_seq_len=MAX_SEQ_LEN, pad_idx=PAD_IDX)
    test_dataset = RealARCDataset(TEST_DIR_SAME, max_seq_len=MAX_SEQ_LEN, pad_idx=PAD_IDX)
    test_cross_dataset = RealARCDataset(TEST_DIR_CROSS, max_seq_len=MAX_SEQ_LEN, pad_idx=PAD_IDX)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_cross_loader = DataLoader(test_cross_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # Initialize the massive d_model=256 architecture
    model = MetaNCA(vocab_size=12, d_model=256, nca_steps=30, pad_idx=PAD_IDX).to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    warmup_scheduler = LinearLR(optimizer, start_factor=0.1, total_iters=10)
    decay_scheduler = ExponentialLR(optimizer, gamma=0.95)
    scheduler = ChainedScheduler([warmup_scheduler, decay_scheduler])

    print("Beginning scaled training protocol...")
    best_acc = 0.0

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device, PAD_IDX, max_grad_norm=1.0)
        scheduler.step() 
        current_lr = optimizer.param_groups[0]['lr']
        
        if epoch % 5 == 0:
            test_acc = evaluate_tasks(model, test_loader, device, PAD_IDX)
            if test_acc > best_acc:
                best_acc = test_acc
                torch.save(model.state_dict(), "best_abstract_nca_large.pth")
                marker = "*"
            else:
                marker = ""
                
            print(f"Epoch {epoch:03d}/{EPOCHS} | LR: {current_lr:.5f} | Loss: {train_loss:.4f} | Same-Concept Acc: {test_acc:.2f}% {marker}")

    # Final Evaluation Block
    print("\n--- Final Cross-Concept Evaluation ---")
    best_model = MetaNCA(vocab_size=12, d_model=256, nca_steps=30, pad_idx=PAD_IDX).to(device)
    best_model.load_state_dict(torch.load("best_abstract_nca_large.pth", map_location=device))
    
    cross_test_acc = evaluate_tasks(best_model, test_cross_loader, device, PAD_IDX)
    
    print(f"========================================")
    print(f" Large Abstract Meta-NCA Same-Concept Acc:  {best_acc:.2f}%")
    print(f" Large Abstract Meta-NCA Cross-Concept Acc: {cross_test_acc:.2f}%")
    print(f"========================================")

Initializing Scaled Abstraction-Wrapped Meta-NCA on: cuda
Loading datasets...
Beginning scaled training protocol...
Epoch 005/150 | LR: 0.00043 | Loss: 0.3705 | Same-Concept Acc: 62.67% *
Epoch 010/150 | LR: 0.00060 | Loss: 0.3231 | Same-Concept Acc: 57.33% 
Epoch 015/150 | LR: 0.00046 | Loss: 0.2908 | Same-Concept Acc: 53.33% 


## Final pipeline: Meta-NCA + abstraction (d_model = 64)
The code that was submitted. It matches Experiment 2 apart from logging. No random seed is set, so results vary from run to run. The run behind the headline numbers (90.67% / 43.05%) is logged in `archive/meta_nca_final_run.ipynb`.

In [5]:
class RealARCDataset(Dataset):
    def __init__(self, json_dir, max_seq_len=30, pad_idx=11, num_support=30):
        self.json_dir = json_dir
        self.files = [f for f in os.listdir(json_dir) if f.endswith('.json')]
        self.max_seq_len = max_seq_len
        self.pad_idx = pad_idx
        self.num_support = num_support

    def __len__(self):
        return len(self.files)

    def flatten_to_1d(self, grid):
        if not isinstance(grid, list):
            return [grid]
        flat_list = []
        for item in grid:
            if isinstance(item, list):
                flat_list.extend(self.flatten_to_1d(item))
            else:
                flat_list.append(item)
        return flat_list

    def pad_sequence(self, seq_raw):
        seq = self.flatten_to_1d(seq_raw)
        if len(seq) >= self.max_seq_len:
            return seq[:self.max_seq_len]
        return seq + [self.pad_idx] * (self.max_seq_len - len(seq))

    def __getitem__(self, idx):
        file_path = os.path.join(self.json_dir, self.files[idx])
        with open(file_path, 'r') as f:
            task_data = json.load(f)
        raw_support = task_data['train']
        augmented_support = (raw_support * 15)[:self.num_support]
        support_in = torch.tensor([self.pad_sequence(pair['input']) for pair in augmented_support], dtype=torch.long)
        support_out = torch.tensor([self.pad_sequence(pair['output']) for pair in augmented_support], dtype=torch.long)
        support_mask = (support_in == self.pad_idx)
        raw_query = task_data['test'][0] 
        query_in = torch.tensor(self.pad_sequence(raw_query['input']), dtype=torch.long)
        query_target = torch.tensor(self.pad_sequence(raw_query['output']), dtype=torch.long)
        query_mask = (query_in == self.pad_idx)
        return {
            'support_in': support_in,
            'support_out': support_out,
            'support_mask': support_mask,
            'query_in': query_in,
            'query_target': query_target,
            'query_mask': query_mask
        }

def apply_abstract_color_mapping(support_in, support_out, query_in, query_target=None, pad_idx=11, bg_idx=0):
    B = support_in.shape[0]
    abs_sup_in = torch.full_like(support_in, bg_idx)
    abs_sup_out = torch.full_like(support_out, bg_idx)
    abs_query_in = torch.full_like(query_in, bg_idx)
    abs_query_target = torch.full_like(query_target, bg_idx) if query_target is not None else None

    for b in range(B):
        task_pixels = support_in[b].flatten()
        unique_colors = torch.unique(task_pixels)
        valid_colors = [c.item() for c in unique_colors if c.item() not in [bg_idx, pad_idx]]
        
        forward_map = {bg_idx: bg_idx, pad_idx: pad_idx}
        for abstract_id, raw_color in enumerate(valid_colors, start=1):
            forward_map[raw_color] = abstract_id
            
        for raw, abstract in forward_map.items():
            abs_sup_in[b][support_in[b] == raw] = abstract
            abs_sup_out[b][support_out[b] == raw] = abstract
            abs_query_in[b][query_in[b] == raw] = abstract
            if abs_query_target is not None:
                abs_query_target[b][query_target[b] == raw] = abstract

    return abs_sup_in, abs_sup_out, abs_query_in, abs_query_target

class SupportEncoder(nn.Module):
    def __init__(self, vocab_size=12, d_model=64, pad_idx=11):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.feature_extractor = nn.Sequential(
            nn.Conv1d(d_model * 2, d_model * 2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(d_model * 2, d_model, kernel_size=3, padding=1),
            nn.AdaptiveMaxPool1d(1) 
        )
        self.rule_projector = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

    def forward(self, support_in, support_out):
        B, N, S = support_in.shape
        emb_in = self.embedding(support_in)
        emb_out = self.embedding(support_out)
        combined = torch.cat([emb_in, emb_out], dim=-1)
        combined = combined.view(B * N, S, -1).transpose(1, 2)
        features = self.feature_extractor(combined).squeeze(-1)
        features = features.view(B, N, -1)
        rule_vector = features.mean(dim=1) 
        return self.rule_projector(rule_vector)


class MetaNCA(nn.Module):
    def __init__(self, vocab_size=12, d_model=64, nca_steps=10, pad_idx=11):
        super().__init__()
        self.d_model = d_model
        self.nca_steps = nca_steps
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.support_encoder = SupportEncoder(vocab_size, d_model, pad_idx)
        self.kernel_size = 3
        weight_params = d_model * d_model * self.kernel_size
        bias_params = d_model
        self.hyper_net = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.ReLU(),
            nn.Linear(d_model * 2, weight_params + bias_params)
        )
        self.output_layer = nn.Linear(d_model, vocab_size)

    def forward(self, support_in, support_out, support_mask, query_in, query_mask, nca_steps=None):
        B, S = query_in.shape
        steps = nca_steps if nca_steps is not None else self.nca_steps
        rule_vector = self.support_encoder(support_in, support_out)
        generated_params = self.hyper_net(rule_vector)
        weight_size = self.d_model * self.d_model * self.kernel_size
        weights_flat = generated_params[:, :weight_size]
        biases_flat = generated_params[:, weight_size:]
        dynamic_weights = weights_flat.reshape(B * self.d_model, self.d_model, self.kernel_size)
        dynamic_biases = biases_flat.reshape(B * self.d_model)
        state = self.embedding(query_in).transpose(1, 2)
        for _ in range(steps):
            state_reshaped = state.reshape(1, B * self.d_model, S)
            update = F.conv1d(
                state_reshaped, weight=dynamic_weights, bias=dynamic_biases, 
                padding=self.kernel_size // 2, groups=B 
            )
            update = update.view(B, self.d_model, S)
            state = state + F.relu(update)
        final_state = state.transpose(1, 2)
        return self.output_layer(final_state)

def train_epoch(model, dataloader, optimizer, criterion, device, pad_idx, max_grad_norm=1.0):
    model.train()
    total_loss = 0
    for batch in dataloader:
        optimizer.zero_grad()
        support_in = batch['support_in'].to(device)
        support_out = batch['support_out'].to(device)
        support_mask = batch['support_mask'].to(device)
        query_in = batch['query_in'].to(device)
        query_target = batch['query_target'].to(device)
        query_mask = batch['query_mask'].to(device)
        abs_sup_in, abs_sup_out, abs_query_in, abs_query_target = apply_abstract_color_mapping(
            support_in, support_out, query_in, query_target, pad_idx=pad_idx
        )
        dynamic_steps = random.randint(8, 12)
        logits = model(
            abs_sup_in, abs_sup_out, support_mask, 
            abs_query_in, query_mask, 
            nca_steps=dynamic_steps
        )
        flat_logits = logits.reshape(-1, logits.size(-1))
        flat_targets = abs_query_target.reshape(-1)
        loss = criterion(flat_logits, flat_targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate_tasks(model, dataloader, device, pad_idx):
    model.eval()
    correct_tasks = 0
    total_tasks = 0
    with torch.no_grad():
        for batch in dataloader:
            support_in = batch['support_in'].to(device)
            support_out = batch['support_out'].to(device)
            support_mask = batch['support_mask'].to(device)
            query_in = batch['query_in'].to(device)
            query_target = batch['query_target'].to(device)
            query_mask = batch['query_mask'].to(device)
            abs_sup_in, abs_sup_out, abs_query_in, abs_query_target = apply_abstract_color_mapping(
                support_in, support_out, query_in, query_target, pad_idx=pad_idx
            )
            logits = model(
                abs_sup_in, abs_sup_out, support_mask, 
                abs_query_in, query_mask, 
                nca_steps=10 
            )
            predictions = logits.argmax(dim=-1) 
            for i in range(len(predictions)):
                valid_mask = ~query_mask[i] 
                valid_preds = predictions[i][valid_mask]
                valid_targets = abs_query_target[i][valid_mask]
                if torch.equal(valid_preds, valid_targets):
                    correct_tasks += 1
                total_tasks += 1
    return (correct_tasks / total_tasks) * 100 if total_tasks > 0 else 0.0

if __name__ == "__main__":
    os.environ['WANDB_MODE'] = 'offline' 
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print("="*60)
    print(f"{'1D-ARC META-NCA INITIALIZATION':^60}")
    print("="*60)
    print(f"[*] Compute Device   : {device}")
    print(f"[*] Architecture     : Meta-Neural Cellular Automata")
    print(f"[*] Mapping Strategy : Dynamic Role-Based Abstraction")
    print("-" * 60)

    TRAIN_DIR = r"../data/1D-ARC/train"
    TEST_DIR_SAME = r"../data/1D-ARC/test_same_concept"
    TEST_DIR_CROSS = r"../data/1D-ARC/test_cross_concept"
    
    BATCH_SIZE = 16
    MAX_SEQ_LEN = 30
    PAD_IDX = 11
    EPOCHS = 100

    print("[*] Loading and formatting dataset partitions...")
    train_dataset = RealARCDataset(TRAIN_DIR, max_seq_len=MAX_SEQ_LEN, pad_idx=PAD_IDX)
    test_dataset = RealARCDataset(TEST_DIR_SAME, max_seq_len=MAX_SEQ_LEN, pad_idx=PAD_IDX)
    test_cross_dataset = RealARCDataset(TEST_DIR_CROSS, max_seq_len=MAX_SEQ_LEN, pad_idx=PAD_IDX)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_cross_loader = DataLoader(test_cross_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = MetaNCA(vocab_size=12, d_model=64, nca_steps=10, pad_idx=PAD_IDX).to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    warmup_scheduler = LinearLR(optimizer, start_factor=0.1, total_iters=10)
    decay_scheduler = ExponentialLR(optimizer, gamma=0.95)
    scheduler = ChainedScheduler([warmup_scheduler, decay_scheduler])

    print("[*] Commencing training protocol...")
    print("-" * 60)
    print(f"{'EPOCH':<10} | {'LR':<10} | {'LOSS':<10} | {'SAME-CONCEPT ACC':<20}")
    print("-" * 60)
    
    best_acc = 0.0

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device, PAD_IDX, max_grad_norm=1.0)
        scheduler.step() 
        current_lr = optimizer.param_groups[0]['lr']
        
        if epoch % 5 == 0:
            test_acc = evaluate_tasks(model, test_loader, device, PAD_IDX)
            marker = ""
            if test_acc > best_acc:
                best_acc = test_acc
                torch.save(model.state_dict(), "best_abstract_nca.pth")
                marker = "(NEW BEST)"
                
            print(f"Epoch {epoch:03d} | {current_lr:.6f} | {train_loss:.4f}   | {test_acc:>6.2f}% {marker}")

    print("\n" + "="*60)
    print(f"{'FINAL CROSS-CONCEPT (OOD) EVALUATION':^60}")
    print("="*60)
    
    best_model = MetaNCA(vocab_size=12, d_model=64, nca_steps=10, pad_idx=PAD_IDX).to(device)
    
    best_model.load_state_dict(torch.load("best_abstract_nca.pth", map_location=device, weights_only=True))
    
    cross_test_acc = evaluate_tasks(best_model, test_cross_loader, device, PAD_IDX)
    
    print(f"[*] Interpolation (Same-Concept) Accuracy   : {best_acc:.2f}%")
    print(f"[*] Extrapolation (Cross-Concept) Accuracy  : {cross_test_acc:.2f}%")
    print("="*60)
    print("Pipeline execution completed successfully.")

               1D-ARC META-NCA INITIALIZATION               
[*] Compute Device   : cuda
[*] Architecture     : Meta-Neural Cellular Automata
[*] Mapping Strategy : Dynamic Role-Based Abstraction
------------------------------------------------------------
[*] Loading and formatting dataset partitions...
[*] Commencing training protocol...
------------------------------------------------------------
EPOCH      | LR         | LOSS       | SAME-CONCEPT ACC    
------------------------------------------------------------
Epoch 005 | 0.000426 | 0.2856   |  72.00% (NEW BEST)
Epoch 010 | 0.000599 | 0.1824   |  77.33% (NEW BEST)
Epoch 015 | 0.000463 | 0.0991   |  82.67% (NEW BEST)
Epoch 020 | 0.000358 | 0.0773   |  89.33% (NEW BEST)
Epoch 025 | 0.000277 | 0.0546   |  85.33% 
Epoch 030 | 0.000215 | 0.0386   |  84.00% 
Epoch 035 | 0.000166 | 0.0263   |  89.33% 
Epoch 040 | 0.000129 | 0.0231   |  86.67% 
Epoch 045 | 0.000099 | 0.0144   |  88.00% 
Epoch 050 | 0.000077 | 0.0131   |  89.33% 
Epoch 